In [1]:
import sys
import pandas as pd

In [2]:
DEFAULT_CSV_PATH = "fb_ads_president_scored_anon.csv"

RANGE_COLUMNS = ["estimated_audience_size", "impressions", "spend"]

In [3]:
def range_to_midpoint(text):
    """Turning "{'lower_bound': '200', 'upper_bound': '299'}" into 249.5.
    Some rows only have one of the two keys, -- in that case we just use whichever number is present."""
    if pd.isna(text):
        return None
    text = str(text).replace("'", "").replace("{", "").replace("}", "")
    lower = None
    upper = None
    for part in text.split(","):
        key, _, value = part.partition(":")
        key = key.strip()
        value = value.strip()
        if value == "":
            continue
        if key == "lower_bound":
            lower = float(value)
        elif key == "upper_bound":
            upper = float(value)
 
    if lower is None and upper is None:
        return None
    if lower is None:
        return upper
    if upper is None:
        return lower
    return (lower + upper) / 2

In [4]:
def load_and_clean(path):
    df = pd.read_csv(path)
    for col in RANGE_COLUMNS:
        df[col] = df[col].apply(range_to_midpoint)
    df["ad_creation_time"] = pd.to_datetime(df["ad_creation_time"])
    return df

In [6]:
def main(path=None):
    if path is None:
        path = DEFAULT_CSV_PATH
 
    df = load_and_clean(path)
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")
    print(df.dtypes)
 
    print("\n--- Missing values ---")
    print(df.isna().sum())
 
    print("\n--- Numeric columns: describe() ---")
    numeric_cols = df.select_dtypes(include="number").columns
    print(df[numeric_cols].describe().T)
 
    print("\n--- Categorical columns: top 5 values each ---")
    text_cols = df.select_dtypes(include="object").columns
    for col in text_cols:
        print(f"\n{col}")
        print(f"unique: {df[col].nunique():,}")
        print(df[col].value_counts().head(5))
 
    #quick side-by-side check against the pure Python script's numbers
    print("\n--- Cross-check: spend/impressions/audience ---")
    for col in RANGE_COLUMNS:
        print(f"\n{col}")
        print(f"count : {df[col].count():,}")
        print(f"mean  : {df[col].mean():.2f}")
        print(f"min   : {df[col].min():.2f}")
        print(f"max   : {df[col].max():.2f}")
        print(f"median: {df[col].median():.2f}")
        print(f"stdev : {df[col].std():.2f}")

In [7]:
if __name__ == "__main__":
    if len(sys.argv) > 1 and not sys.argv[1].startswith("-"):
        main(sys.argv[1])
    else:
        main()

Shape: 246,745 rows x 40 columns

page_id                                              object
page_name                                            object
ad_id                                                object
ad_creation_time                             datetime64[ns]
ad_delivery_start_time                               object
ad_delivery_stop_time                                object
bylines                                              object
currency                                             object
estimated_audience_size                             float64
impressions                                         float64
spend                                               float64
publisher_platforms                                  object
illuminating_scored_message                          object
illuminating_mentions                                object
illuminating_scam                                     int64
illuminating_election_integrity_Truth                 int64
illumi